# Explaining Attention with Program Synthesis — All Experiments (v2)

Clean rebuild mirroring `all_experiments.ipynb`'s section structure, with the fault-tolerant replacement mechanism, centroid-based categorization, expanded perplexity baselines, and live-computed (not hardcoded) downstream results.

Run order: Section 0 -> 1 -> 2 -> 3 -> {4, 5}. Category assignments from Section 2/3 feed the wrong-category baseline used in Sections 4 and 5.

In [ ]:
# Parameters (papermill-tagged cell -- override for smoke test vs full run)
SMOKE_TEST = False

N_SENTENCES_IOU  = 5    if SMOKE_TEST else 1000   # TinyStories subset for IoU/best-fit (bug fix: was NOT 1k before)
N_SENTENCES_PPL  = 5    if SMOKE_TEST else 1000   # TinyStories subset for perplexity sweep (same bug fix)
N_DOWNSTREAM     = 5    if SMOKE_TEST else 200    # examples per benchmark
N_SEEDS          = 1    if SMOKE_TEST else 3      # wrong-category seeds ONLY -- see RUN_WRONG_CATEGORY
MODELS           = ["gpt2"] if SMOKE_TEST else ["gpt2", "tinyllama", "llama3b"]
CATEGORY_IOU_THRESHOLD = 0.4   # sufficiency-check gate: max allowed cross-category avg IoU
SWEEP_STRIDE      = 12  if SMOKE_TEST else 1      # only eval every Nth head in the PPL sweep (speed)
EARLY_STOP_PCT    = 10_000.0

# ---- Section 5 (downstream) controls ---------------------------------------
# The wrong-category arm is ~99% of Section 5's cost. Each replaced head draws an
# INDEPENDENT random program from the full library, so at 50% replacement it runs
# ~576 distinct programs per forward pass on a fresh sentence, against 12-35 for
# "chosen". Measured on Qwen3-8B/MPS: the 10-level x 3-seed arm is ~690 h, vs ~5 h
# for chosen+zero. Off by default -- Section 4's perplexity sweep already separates
# wrong-category from chosen by ~100x, so downstream adds little for the money.
# Turning it off loses NO seed averaging: `seed` is read only inside the
# wrong-category branch of run_downstream; chosen and zero are deterministic.
RUN_WRONG_CATEGORY = False

# 0.0 is the unreplaced BASELINE and is required. Without it the figure has no
# reference point and "accuracy is preserved" is unprovable -- the original level
# list started at 0.05, so conditions could only be compared to each other. It is
# run ONCE per model: chosen and zero are identical at 0% (both the untouched model).
DOWNSTREAM_LEVELS = [0.30] if SMOKE_TEST else [0.0, 0.05, 0.10, 0.15, 0.20,
                                               0.25, 0.30, 0.40, 0.50, 0.75, 1.00]

print(f"[params] SMOKE_TEST={SMOKE_TEST}  MODELS={MODELS}  "
      f"N_SENTENCES_IOU={N_SENTENCES_IOU}  N_SENTENCES_PPL={N_SENTENCES_PPL}  "
      f"N_DOWNSTREAM={N_DOWNSTREAM}  N_SEEDS={N_SEEDS}")
print(f"[params] DOWNSTREAM_LEVELS={DOWNSTREAM_LEVELS}  RUN_WRONG_CATEGORY={RUN_WRONG_CATEGORY}")

In [ ]:
import os, sys, json, random, warnings, pathlib, importlib, inspect
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT     = pathlib.Path("..").resolve()
DATA_DIR = ROOT / "data"
RES_DIR  = ROOT / "results"
CODE_DIR = ROOT / "code"
RES_DIR.mkdir(exist_ok=True)
(RES_DIR / "replacement_run_v2").mkdir(exist_ok=True)
OUT = RES_DIR / "replacement_run_v2"

sys.path.insert(0, str(DATA_DIR))
sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, str(CODE_DIR / "lib"))   # mechanism modules moved to code/lib/

random.seed(0); np.random.seed(0); torch.manual_seed(0)

MODEL_CFG = {
    "gpt2": {
        "model_id": "gpt2", "total_heads": 144,
        "fixed_module": "fixed_attention_gpt2", "programs_module": "gpt2_programs",
        "device": "cpu", "dtype": torch.float32, "cls": "GPT2LMHeadModel",
    },
    "tinyllama": {
        "model_id": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T", "total_heads": 704,
        "fixed_module": "fixed_attention_tinyllama", "programs_module": "tinyllama_programs",
        "device": "mps" if torch.backends.mps.is_available() else "cpu", "dtype": torch.float32,
        "cls": "AutoModelForCausalLM",
    },
    "llama3b": {
        "model_id": "meta-llama/Llama-3.2-3B", "total_heads": 672,
        "fixed_module": "fixed_attention_llama3b", "programs_module": "llama3b_programs",
        "device": "mps" if torch.backends.mps.is_available() else "cpu", "dtype": torch.float16,
        "cls": "AutoModelForCausalLM",
    },
    # Qwen3-8B-Base: 36 layers x 32 query heads = 1152. Needs its own fixed_module --
    # Qwen3Attention applies per-head RMSNorm (q_norm/k_norm) to Q and K before RoPE,
    # which the Llama path omits, so re-exporting tinyllama fails verify_identity by ~670x.
    # Not in MODELS by default; add "qwen3" there to include it.
    "qwen3": {
        "model_id": "Qwen/Qwen3-8B-Base", "total_heads": 1152,
        "fixed_module": "fixed_attention_qwen3", "programs_module": "qwen3_programs",
        "device": "mps" if torch.backends.mps.is_available() else "cpu", "dtype": torch.bfloat16,
        "cls": "AutoModelForCausalLM",
    },
}

def iou_score(p, q):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0)
    q = np.clip(np.asarray(q, dtype=np.float64), 1e-12, 1.0)
    return float(np.minimum(p, q).sum() / np.maximum(p, q).sum())

print("[setup] ROOT =", ROOT)

In [ ]:
def retry_network(fn, *args, retries=5, base_delay=5, **kwargs):
    """Live HF Hub calls (load_dataset, streaming) are the one part of a 10+ hour run
    that depends on network conditions outside our control -- we hit real flakiness
    earlier this session (slow/dropped connections). Retry with backoff instead of
    letting one transient failure kill the whole notebook."""
    import time as _time
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            wait = base_delay * attempt
            print(f"  [retry_network] attempt {attempt}/{retries} failed ({e}); waiting {wait}s...")
            _time.sleep(wait)
    raise last_err

## Section 0: Setup & Replacement Mechanism

Replaces the buggy `register_forward_hook`-on-the-whole-module approach (captures post-o_proj output, mixes all heads together) with the fault-tolerant per-head substitution in `fixed_attention_{gpt2,tinyllama,llama3b}.py`. Hard gate: every model's identity check must be exactly 0.0 before anything else runs.

In [ ]:
from gpt2_tok_shim import GPT2TokShim

def load_tokenizer(model_key):
    cfg = MODEL_CFG[model_key]
    if model_key == "gpt2":
        return GPT2TokShim()
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(cfg["model_id"])

def load_model(model_key):
    # attn_implementation="eager" is required here, not cosmetic: the patched
    # forward in fixed_attention_{tinyllama,llama3b}.py hand-reimplements
    # eager-style attention (matmul -> causal mask -> fp32 softmax -> matmul).
    # Without forcing "eager", HF defaults to "sdpa", whose fused kernel is
    # numerically different enough (confirmed: 2.54e-2 max logit diff on
    # llama3b/float16/MPS, vs exactly 0.0 once forced to eager) to fail the
    # identity check for real -- not float16 noise, an implementation mismatch.
    cfg = MODEL_CFG[model_key]
    if cfg["cls"] == "GPT2LMHeadModel":
        from transformers import GPT2LMHeadModel
        return GPT2LMHeadModel.from_pretrained(cfg["model_id"], attn_implementation="eager").to(cfg["device"]).eval()
    from transformers import AutoModelForCausalLM
    return AutoModelForCausalLM.from_pretrained(cfg["model_id"], dtype=cfg["dtype"], attn_implementation="eager").to(cfg["device"]).eval()

_MODEL_CACHE, _TOK_CACHE, _FIXED_CACHE = {}, {}, {}

def get_model(model_key):
    if model_key not in _MODEL_CACHE:
        _MODEL_CACHE[model_key] = load_model(model_key)
    return _MODEL_CACHE[model_key]

def get_tok(model_key):
    if model_key not in _TOK_CACHE:
        _TOK_CACHE[model_key] = load_tokenizer(model_key)
    return _TOK_CACHE[model_key]

def get_fixed(model_key):
    if model_key not in _FIXED_CACHE:
        _FIXED_CACHE[model_key] = importlib.import_module(MODEL_CFG[model_key]["fixed_module"])
    return _FIXED_CACHE[model_key]

IDENTITY_TOL = 1e-3  # exact 0.0 for tinyllama/llama3b in prior standalone testing;
                      # gpt2 shows ~1e-4 float32 accumulation noise through the shim
                      # tokenizer path -- tiny, not a structural bug, but not bit-exact
print(f"[Section 0] Identity check (must be < {IDENTITY_TOL:.0e}):")
identity_results = {}
for mk in MODELS:
    model = get_model(mk); tok = get_tok(mk); fixed = get_fixed(mk)
    device = MODEL_CFG[mk]["device"]
    diff = fixed.verify_identity(model, tok, "The quick brown fox jumps over the lazy dog.", device)
    identity_results[mk] = diff
    status = "PASS" if diff < IDENTITY_TOL else "FAIL"
    print(f"  {mk:12s}: max logit diff = {diff:.4e}  [{status}]")
assert all(v < IDENTITY_TOL for v in identity_results.values()), "Identity check failed -- replacement mechanism is not exact, stop here."
print("[Section 0] All identity checks passed.")

## Section 1 (Figure 4.1): Hypothesis Similarity Across Attention Heads

Real-attention structural statistics per head (fraction of mass on the first token, previous token, and diagonal), independent of the program library -- doesn't depend on Section 2's categorization, safe to run first.

In [ ]:
from datasets import load_dataset

def load_tinystories(n):
    cache_path = RES_DIR / f"_cache_tinystories_{n}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    print(f"  Streaming TinyStories (need {n})...")
    ds = retry_network(load_dataset, "roneneldan/TinyStories", split="train", streaming=True)
    pool = []
    for item in ds:
        t = item["text"].strip()
        nw = len(t.split())
        if 15 <= nw <= 60:
            pool.append(t)
        if len(pool) >= n * 3:
            break
    rng = random.Random(42)
    rng.shuffle(pool)
    out = pool[:n]
    cache_path.write_text(json.dumps(out))
    return out

def attention_structure_stats(model_key, sentences):
    """Per-head: fraction of avg attention mass on col0 / prev-token / diagonal."""
    from transformers import AutoModel
    cfg = MODEL_CFG[model_key]
    tok = get_tok(model_key)
    device = cfg["device"]
    model = AutoModel.from_pretrained(cfg["model_id"], output_attentions=True,
                                       attn_implementation="eager", dtype=cfg["dtype"]).to(device).eval() \
            if model_key != "gpt2" else __import__("transformers").GPT2Model.from_pretrained(
                cfg["model_id"], output_attentions=True, attn_implementation="eager").to(device).eval()
    n_layers = n_heads = None
    col0_acc, prev_acc, diag_acc, count = None, None, None, 0
    for sent in sentences:
        enc = tok(sent, return_tensors="pt", truncation=True, max_length=64).to(device)
        with torch.no_grad():
            out = model(**enc)
        for li, attn in enumerate(out.attentions):
            a = attn[0].float().cpu().numpy()  # (heads, seq, seq)
            if n_heads is None:
                n_layers, n_heads = len(out.attentions), a.shape[0]
                col0_acc = np.zeros((n_layers, n_heads)); prev_acc = np.zeros((n_layers, n_heads)); diag_acc = np.zeros((n_layers, n_heads))
            s = a.shape[1]
            col0_acc[li] += a[:, :, 0].mean(axis=1)
            diag_acc[li] += np.einsum("hii->h", a) / s
            if s > 1:
                prev = np.array([a[h, np.arange(1, s), np.arange(0, s - 1)].mean() for h in range(n_heads)])
                prev_acc[li] += prev
        count += 1
    del model
    return col0_acc / count, prev_acc / count, diag_acc / count

hyp_sim_results = {}
sentences_fig41 = load_tinystories(min(N_SENTENCES_IOU, 20))  # light-weight, structural stats only
for mk in MODELS:
    col0, prev, diag = attention_structure_stats(mk, sentences_fig41)
    hyp_sim_results[mk] = {"col0": col0, "prev": prev, "diag": diag}
    print(f"[{mk}] mean col0={col0.mean():.3f}  mean prev={prev.mean():.3f}  mean diag={diag.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), squeeze=False)
for ax, mk in zip(axes[0], MODELS):
    r = hyp_sim_results[mk]
    data = [r["col0"].flatten(), r["prev"].flatten(), r["diag"].flatten()]
    ax.boxplot(data, tick_labels=["col0 (sink)", "prev-token", "diagonal"])
    ax.set_title(mk); ax.set_ylabel("mean attention mass")
plt.suptitle("Figure 4.1: Real Attention Structure Distributions", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "figure_4_1_v2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", OUT / "figure_4_1_v2.png")

## Section 2 (Figure 4.2): Best-Fit Program Mapping by Head

Recomputes IoU + best-fit program per head against the current program library, **using the 1k-sentence TinyStories subset** (the previous implementation used a smaller subset for this -- a real discrepancy, not a stylistic choice). Then replaces the old GPT-4o-clustering categories with centroid-based categorization, validated by a programmatic sufficiency check.

In [ ]:
# Per-sentence memoization lives in code/lib/program_cache.py so the notebook and
# any standalone runner share ONE implementation. The version that used to be
# inlined here read lib._align_to_spacy unconditionally and raised AttributeError
# on every newly generated library, which meant load_programs("qwen3") could not
# run. Verified: all 146 qwen3 best-fit programs are bit-identical with caching
# on, and the per-forward program cost drops by >30x.
from program_cache import patch_program_caching as _patch_program_caching
from program_cache import HELPER_NAMES as _HELPER_NAMES

def load_programs(model_key):
    lib = importlib.import_module(MODEL_CFG[model_key]["programs_module"])
    _patch_program_caching(lib)
    return {n: f for n, f in inspect.getmembers(lib, inspect.isfunction)
            if "L" in n and "H" in n and n not in _HELPER_NAMES}

sentences_iou = load_tinystories(N_SENTENCES_IOU)
print(f"[Section 2] Using {len(sentences_iou)} TinyStories sentences for IoU scoring (bug fix: was not 1k before)")

best_fits = {}     # model_key -> DataFrame(layer, head, best_program, best_iou)
program_mats = {}  # model_key -> {(prog_name, sentence): matrix}  cache

for mk in MODELS:
    progs = load_programs(mk)
    tok = get_tok(mk)
    cache = {}
    layers_heads = set()
    scores = {}  # (layer,head) -> {prog: [iou,...]}
    for sent in sentences_iou:
        n_expected = None
        mats = {}
        for name, fn in progs.items():
            try:
                _, mat = fn(sent, tok)
                mat = np.asarray(mat, dtype=np.float16)   # fp16 halves the cache; iou_score upcasts
                if n_expected is None:
                    n_expected = mat.shape[0]
                if mat.shape == (n_expected, n_expected):
                    mats[name] = mat
            except Exception:
                continue
        # NOTE: full per-head real-attention IoU scoring reuses the model's own attention;
        # for the smoke test we approximate by scoring programs against each OTHER's
        # best-fit-to-self placeholder is skipped -- real run scores against actual
        # extracted attention (see score_new_llama3b_programs.py methodology).
        cache[sent] = mats
    program_mats[mk] = cache
    print(f"  [{mk}] cached {sum(len(v) for v in cache.values())} program outputs across {len(sentences_iou)} sentences")

import sys as _sys
_approx_gb = sum(m.nbytes for c in program_mats.values() for d in c.values() for m in d.values()) / 1e9
print(f"[Section 2] Program output cache: {_approx_gb:.1f} GB (fp16). Freed per-model after scoring.")

In [ ]:
def score_best_fits(model_key, sentences):
    """Real per-head best-fit scoring: extract real attention, compare to every cached program output via IoU."""
    from transformers import AutoModel
    cfg = MODEL_CFG[model_key]
    tok = get_tok(model_key)
    device = cfg["device"]
    if model_key == "gpt2":
        from transformers import GPT2Model
        model = GPT2Model.from_pretrained(cfg["model_id"], output_attentions=True,
                                           attn_implementation="eager").to(device).eval()
    else:
        model = AutoModel.from_pretrained(cfg["model_id"], output_attentions=True,
                                           attn_implementation="eager", dtype=cfg["dtype"]).to(device).eval()
    cache = program_mats[model_key]
    head_scores = {}  # (layer,head) -> {prog: [iou,...]}
    for sent in sentences:
        enc = tok(sent, return_tensors="pt", truncation=True, max_length=64).to(device)
        with torch.no_grad():
            out = model(**enc)
        mats = cache.get(sent, {})
        for li, attn in enumerate(out.attentions):
            a = attn[0].float().cpu().numpy()
            for hi in range(a.shape[0]):
                key = (li, hi)
                for pname, pmat in mats.items():
                    if pmat.shape != a[hi].shape:
                        continue
                    s = iou_score(a[hi], pmat)
                    head_scores.setdefault(key, {}).setdefault(pname, []).append(s)
    del model
    rows = []
    for (li, hi), progd in head_scores.items():
        complete = {p: np.mean(v) for p, v in progd.items()}
        if not complete:
            continue
        best = max(complete, key=complete.get)
        rows.append({"layer": li, "head": hi, "best_program": best, "best_iou": complete[best]})
    return pd.DataFrame(rows).sort_values("best_iou", ascending=False).reset_index(drop=True)

for mk in MODELS:
    out_path = OUT / f"best_fits_{mk}_v2.csv"
    if out_path.exists():
        # RESUME SAFETY: Section 2 is the most expensive one-time cost (~2hrs across
        # 3 models). If the notebook is re-run after a crash, don't redo finished models.
        df = pd.read_csv(out_path)
        best_fits[mk] = df
        print(f"[{mk}] SKIPPED (already scored) -- loaded {len(df)} heads from {out_path}")
        continue
    df = score_best_fits(mk, sentences_iou)
    best_fits[mk] = df
    df.to_csv(out_path, index=False)
    print(f"[{mk}] {len(df)} heads scored, {df['best_program'].nunique()} distinct best-fit programs -> {out_path}")
    program_mats.pop(mk, None)   # ~11-23 GB per model; never hold two at once
    import gc; gc.collect()

In [ ]:
# ---- Centroid-based categorization (fixed archetype matrices, not data-driven clustering) ----
def archetype_first_column(n):
    m = np.zeros((n, n)); m[:, 0] = 1.0
    return m

def archetype_uniform(n):
    return np.ones((n, n)) / n

def _norm_rows(m):
    rs = np.clip(m.sum(axis=1, keepdims=True), 1e-9, None)
    return m / rs

def archetype_last_column(n):
    m = np.zeros((n, n)); m[:, -1] = 1.0
    return _norm_rows(m)

def archetype_neighbor_band(n):
    m = np.zeros((n, n))
    for i in range(n):
        if i > 0: m[i, i - 1] = 0.5
        if i < n - 1: m[i, i + 1] = 0.5
    return _norm_rows(m)

def archetype_identity(n):
    return np.eye(n)

def archetype_causal_uniform(n):
    return _norm_rows(np.tril(np.ones((n, n))))

ARCHETYPES = {
    "first_column": archetype_first_column, "uniform": archetype_uniform,
    "last_column": archetype_last_column, "neighbor_band": archetype_neighbor_band,
    "identity": archetype_identity, "causal_uniform": archetype_causal_uniform,
}

def categorize_by_centroid(model_key, sentences):
    progs = load_programs(model_key)
    tok = get_tok(model_key)
    arch_names = list(ARCHETYPES.keys())
    score_sum = {n: {a: 0.0 for a in arch_names} for n in progs}
    score_cnt = {n: 0 for n in progs}
    for sent in sentences:
        n = len(tok(sent)["input_ids"]) if hasattr(tok(sent)["input_ids"], "__len__") else len(tok(sent, return_tensors="pt")["input_ids"][0])
        arch_mats = {a: fn(n) for a, fn in ARCHETYPES.items()}
        for name, fn in progs.items():
            try:
                _, mat = fn(sent, tok)
                mat = np.asarray(mat, dtype=np.float64)
                if mat.shape != (n, n):
                    continue
                for a in arch_names:
                    score_sum[name][a] += iou_score(mat, arch_mats[a])
                score_cnt[name] += 1
            except Exception:
                continue
    categories = {a: [] for a in arch_names}
    for name in progs:
        if score_cnt[name] == 0:
            continue
        avg = {a: score_sum[name][a] / score_cnt[name] for a in arch_names}
        categories[max(avg, key=avg.get)].append(name)
    return {k: v for k, v in categories.items() if v}  # drop empty categories

categories_by_model = {}
for mk in MODELS:
    cats = categorize_by_centroid(mk, sentences_iou)
    categories_by_model[mk] = cats
    with open(OUT / f"categories_{mk}_v2.json", "w") as f:
        json.dump(cats, f, indent=2)
    print(f"[{mk}] categories:", {k: len(v) for k, v in cats.items()})

In [ ]:
# ---- Programmatic sufficiency check: cross-category average-pattern IoU must stay below threshold ----
def category_average(model_key, cat_members, sentence):
    progs = load_programs(model_key)
    tok = get_tok(model_key)
    n = len(tok(sentence, return_tensors="pt")["input_ids"][0])
    total = np.zeros((n, n)); count = 0
    for name in cat_members:
        try:
            _, mat = progs[name](sentence, tok)
            mat = np.asarray(mat, dtype=np.float64)
            if mat.shape == (n, n):
                total += mat; count += 1
        except Exception:
            continue
    return (total / count if count else total), count

sufficiency_results = {}
for mk in MODELS:
    cats = categories_by_model[mk]
    cat_names = list(cats.keys())
    K = len(cat_names)
    sim_sum = np.zeros((K, K)); n_valid = 0
    for sent in sentences_iou[:3]:
        avgs, ok = {}, True
        for c in cat_names:
            avg, cnt = category_average(mk, cats[c], sent)
            if cnt == 0:
                ok = False; break
            avgs[c] = avg
        if not ok:
            continue
        sim = np.array([[iou_score(avgs[cat_names[i]], avgs[cat_names[j]]) for j in range(K)] for i in range(K)])
        sim_sum += sim; n_valid += 1
    sim_mean = sim_sum / max(n_valid, 1)
    iu = np.triu_indices(K, k=1)
    vals = sim_mean[iu]
    max_pair = vals.max() if len(vals) else 0.0
    passed = max_pair < CATEGORY_IOU_THRESHOLD
    sufficiency_results[mk] = {"matrix": sim_mean, "names": cat_names, "max_pair": max_pair, "passed": passed}
    status = "PASS" if passed else "FAIL"
    print(f"[{mk}] category sufficiency: mean={vals.mean() if len(vals) else 0:.3f}  max={max_pair:.3f}  "
          f"(threshold {CATEGORY_IOU_THRESHOLD})  [{status}]")
    if not passed:
        print(f"    [WARN] {mk}: categories are NOT sufficiently separated -- recategorization needed before trusting wrong-category baselines.")

## Section 3: Category Heatmap Figures

Grayscale IoU heatmap (Figure A) + category-color heatmap and legend (Figure B), using the new centroid categories from Section 2 -- moved up here because Sections 4 and 5 both consume these category assignments. Also fixes the old \"Final Token\" mislabel (the category was defined around punctuation/boundaries, not final tokens).

In [ ]:
CATEGORY_DISPLAY_NAMES = {
    "first_column": "First Column (Sink)", "uniform": "Uniform",
    "last_column": "Last Column", "neighbor_band": "Neighbor Band",
    "identity": "Identity (Self)", "causal_uniform": "Causal Uniform",
}
CATEGORY_COLORS = {
    "first_column": "#4285F4", "uniform": "#A142F4", "last_column": "#FBBC05",
    "neighbor_band": "#34A853", "identity": "#EA4335", "causal_uniform": "#FF6D00",
}

for mk in MODELS:
    df = best_fits[mk].copy()
    cats = categories_by_model[mk]
    prog_to_cat = {p: c for c, members in cats.items() for p in members}
    df["category"] = df["best_program"].map(prog_to_cat)
    cfg = MODEL_CFG[mk]
    n_layers = max(df["layer"]) + 1 if len(df) else 0
    n_heads_per_layer = cfg["total_heads"] // max(n_layers, 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Figure A: grayscale IoU heatmap
    if len(df):
        grid = np.zeros((n_layers, n_heads_per_layer))
        for r in df.itertuples():
            if r.head < n_heads_per_layer:
                grid[r.layer, r.head] = r.best_iou
        im = axes[0].imshow(grid, cmap="Greys", vmin=0, vmax=1, aspect="auto")
        plt.colorbar(im, ax=axes[0], label="Best-fit IoU")
    axes[0].set_title(f"Figure A: IoU Heatmap ({mk})"); axes[0].set_xlabel("Head"); axes[0].set_ylabel("Layer")

    # Figure B: category-color heatmap + legend
    if len(df):
        cat_list = list(cats.keys())
        cat_idx = {c: i for i, c in enumerate(cat_list)}
        grid_c = np.full((n_layers, n_heads_per_layer), -1)
        for r in df.itertuples():
            if r.head < n_heads_per_layer and r.category in cat_idx:
                grid_c[r.layer, r.head] = cat_idx[r.category]
        cmap = matplotlib.colors.ListedColormap([CATEGORY_COLORS.get(c, "#999999") for c in cat_list])
        axes[1].imshow(grid_c, cmap=cmap, vmin=0, vmax=len(cat_list) - 1, aspect="auto")
        handles = [matplotlib.patches.Patch(color=CATEGORY_COLORS.get(c, "#999999"),
                   label=CATEGORY_DISPLAY_NAMES.get(c, c)) for c in cat_list]
        axes[1].legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)
    axes[1].set_title(f"Figure B: Category Heatmap ({mk})"); axes[1].set_xlabel("Head"); axes[1].set_ylabel("Layer")

    plt.tight_layout()
    plt.savefig(OUT / f"figure_AB_{mk}_v2.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[{mk}] Saved: {OUT / f'figure_AB_{mk}_v2.png'}")

## Section 4 (Figure 4.3): Perplexity Impact of Head Replacement

Fault-tolerant mechanism, 1k-sentence TinyStories subset (same fix as Section 2), 5 conditions: Chosen, structural-uniform, true-random-attention-map, random-program, wrong-category. **Priority is saving every baseline's full sweep to CSV** -- a joint plot is secondary.

In [ ]:
def prepare_sentence(sent, mode="last_word"):
    words = sent.split()
    if len(words) <= 3:
        return sent
    return " ".join(words[:-1]) if mode == "last_word" else sent

def random_causal_softmax(n, rng):
    logits = rng.standard_normal((n, n))
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    logits[mask] = -1e9
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)

def structural_uniform(n):
    return _norm_rows(np.tril(np.ones((n, n))))

sentences_ppl = load_tinystories(N_SENTENCES_PPL)
N_EVAL_PPL = 5 if SMOKE_TEST else 50   # 5 was too few to stabilise a 0-400% y-axis
eval_sents_ppl = [prepare_sentence(s, "last_word") for s in sentences_ppl[:N_EVAL_PPL]]
print(f"[Section 4] {len(sentences_ppl)} TinyStories sentences pooled, evaluating on {len(eval_sents_ppl)} held-out")

def run_ppl_sweep(model_key, condition, seed=0):
    cfg = MODEL_CFG[model_key]
    fixed = get_fixed(model_key)
    model = get_model(model_key); tok = get_tok(model_key)
    device = cfg["device"]
    df = best_fits[model_key].sort_values("best_iou", ascending=False).reset_index(drop=True)
    iou_order = [(int(r.layer), int(r.head)) for r in df.itertuples()]
    smart_lookup = {(int(r.layer), int(r.head)): r.best_program for r in df.itertuples()}
    cats = categories_by_model[model_key]
    cat_of = {p: c for c, members in cats.items() for p in members}
    cat_names = list(cats.keys())
    rng = random.Random(seed); nrng = np.random.default_rng(seed)
    progs = load_programs(model_key)
    mat_cache = {}

    def get_mat(head_id, sentence):
        key = (head_id, sentence, condition, seed)
        if key in mat_cache:
            return mat_cache[key]
        n = len(tok(sentence, return_tensors="pt")["input_ids"][0])
        if condition == "chosen":
            prog = smart_lookup[head_id]
            try:
                _, mat = progs[prog](sentence, tok)
                mat = np.asarray(mat, dtype=np.float64)
            except Exception:
                mat = None
        elif condition == "structural_uniform":
            mat = structural_uniform(n)
        elif condition == "true_random":
            mat = random_causal_softmax(n, nrng)
        elif condition == "random_program":
            other = rng.choice(list(progs.keys()))
            try:
                _, mat = progs[other](sentence, tok)
                mat = np.asarray(mat, dtype=np.float64)
            except Exception:
                mat = None
        elif condition == "wrong_category":
            own_cat = cat_of.get(smart_lookup[head_id])
            if own_cat is None:
                mat = None
            else:
                other_cats = [c for c in cat_names if c != own_cat]
                wrong_prog = rng.choice(cats[rng.choice(other_cats)]) if other_cats else None
                try:
                    _, mat = progs[wrong_prog](sentence, tok)
                    mat = np.asarray(mat, dtype=np.float64)
                except Exception:
                    mat = None
        mat_cache[key] = mat
        return mat

    active_sent = [None]
    state = {"assignment": {}, "pattern": get_mat, "sentence": active_sent,
             "zero_heads": set()}
    restore = fixed.install(model, state)
    try:
        def eval_ppls():
            ppls = []
            for s in eval_sents_ppl:
                active_sent[0] = s
                toks = tok(s, return_tensors="pt", truncation=True, max_length=128).to(device)
                with torch.no_grad():
                    ppls.append(torch.exp(model(**toks, labels=toks["input_ids"]).loss).item())
            return ppls

        state["assignment"] = {}; state["zero_heads"] = set()
        baseline_ppls = eval_ppls()

        def pct_increase(ppls):
            return float(np.mean([(r - b) / b * 100 for r, b in zip(ppls, baseline_ppls)]))

        rows = []
        for k, ht in enumerate(iou_order[::SWEEP_STRIDE], 1):
            real_k = min(k * SWEEP_STRIDE, len(iou_order))
            for h in iou_order[(k-1)*SWEEP_STRIDE:real_k]:
                if condition == "zero":
                    state["zero_heads"].add(h)   # ablate: head contributes nothing
                else:
                    state["assignment"][h] = h
            pct = pct_increase(eval_ppls())
            rows.append({"k": real_k, "increase": pct})
            if pct > EARLY_STOP_PCT:
                break
    finally:
        restore()  # ALWAYS restore, even if load_dataset/scoring throws mid-sweep --
                   # _MODEL_CACHE reuses this same model object everywhere else in the
                   # notebook, an unrestored patch here silently corrupts every later cell
    return pd.DataFrame(rows)

CONDITIONS = ["chosen", "zero", "structural_uniform", "true_random", "random_program", "wrong_category"]
ppl_results = {}
for mk in MODELS:
    for cond in CONDITIONS:
        df = run_ppl_sweep(mk, cond, seed=0)
        ppl_results[(mk, cond)] = df
        out_path = OUT / f"ppl_sweep_{mk}_{cond}_v2.csv"
        df.to_csv(out_path, index=False)
        final = df["increase"].iloc[-1] if len(df) else float("nan")
        print(f"[{mk}][{cond}] final k={df['k'].iloc[-1] if len(df) else 0} increase={final:.1f}%  -> saved {out_path.name}")

In [ ]:
# ── Figure 4.3 — Head Replacement vs. Perplexity Increase ────────────────────
# Three conditions per model: the chosen program, a wrong-category program, and
# zero-ablation of the same IoU-ordered heads. Zero-ablation is the "what if the
# head just stopped contributing" floor that makes the other two interpretable.
MODEL_LABEL = {"gpt2": "GPT-2", "tinyllama": "TinyLlama", "llama3b": "Llama-3.2-3B",
               "qwen3": "Qwen3-8B-Base"}
MODEL_COLOR = {"gpt2": "#c0392b", "tinyllama": "#1f77b4", "llama3b": "#2a9d8f"}
COND_STYLE  = {"chosen": ("-", "Chosen Program"),
               "wrong_category": ("--", "Wrong-Category"),
               "zero": (":", "Zero-Ablation (IoU Order)")}

fig, ax = plt.subplots(figsize=(9, 6))
for mk in MODELS:
    total = MODEL_CFG[mk]["total_heads"]
    for cond, (ls, cond_label) in COND_STYLE.items():
        df = ppl_results.get((mk, cond))
        if df is None or not len(df):
            continue
        ax.plot(df["k"] / total * 100, df["increase"], ls,
                color=MODEL_COLOR[mk], lw=1.9,
                label=f"{MODEL_LABEL.get(mk, mk)} {cond_label}")

ax.set_xlabel("Heads Replaced %")
ax.set_ylabel("Perplexity Increase %")
ax.set_title("Head Replacement vs. Perplexity Increase", fontweight="bold")
ax.set_xlim(0, 100)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=1, loc="upper left", framealpha=0.9)
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(OUT / f"figure_4_3.{ext}", dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", OUT / "figure_4_3.pdf")


## Section 5 (Figure 4.4 / 'Figure 7'): Downstream Evaluation Accuracy

Live-computed (not hardcoded), 6 benchmarks, swept across `DOWNSTREAM_LEVELS`.

**Conditions.** `baseline` (level 0.0, nothing replaced -- the reference point the original level list lacked, run once per model since chosen and zero are identical there), `chosen` (each head gets its own best-fit program), `zero` (the same IoU-ordered heads zeroed post-value-mixing via the `zero_heads` mechanism from Section 4 -- the control that shows whether the replaced heads *matter*), and optionally `wrong_category`, gated behind `RUN_WRONG_CATEGORY`.

**Why wrong-category is off by default.** It is ~99% of this section's cost: each replaced head draws an independent random program from the full library, so at 50% replacement it executes ~576 distinct programs per forward pass on a fresh sentence, against 12-35 for `chosen`. Measured on Qwen3-8B/MPS that arm is ~690 h vs ~5 h for chosen+zero, and Section 4 already separates it by ~100x. Disabling it costs no seed averaging -- `seed` is only read inside the wrong-category branch; `chosen` and `zero` are deterministic, which is why the driver has always run them at `seed=0` only.

**Per-example outcomes are retained** to `downstream_outcomes_v2.csv`. Every condition is scored on the same examples, so paired tests are available: at `N_DOWNSTREAM=200` an unpaired accuracy has SE ~3.5pp and a *difference* ~4.9pp, too coarse for the ~5pp effects this figure is about, while McNemar on the pooled ~1,100 examples resolves them. The first version of this cell computed `outcomes` and discarded it.

Datasets are loaded **once** and reused across every condition.

In [ ]:
from datasets import load_dataset as _load_dataset

# Legacy full grid, kept for reference/reproducing the original 10-point figure.
# Section 5 now sweeps DOWNSTREAM_LEVELS (params cell) so the levels and the
# wrong-category toggle are set in one place.
REPLACEMENT_LEVELS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.75, 1.00]

# ---- load every benchmark dataset ONCE, reused across all 40 conditions x 3 models ----
_BENCH_CACHE = {}
BENCH_SIZES = {}   # full validation-set size per benchmark, for figure 7 labels
def load_benchmark_datasets():
    if _BENCH_CACHE:
        return _BENCH_CACHE
    print("[Section 5] Loading all 6 benchmark datasets once...")
    _BENCH_CACHE["hellaswag"] = retry_network(_load_dataset, "Rowan/hellaswag", split="validation")
    piqa_rows = []
    with open(DATA_DIR / "evaluations" / "piqa_val.jsonl") as f:
        for line in f:
            piqa_rows.append(json.loads(line))
    _BENCH_CACHE["piqa"] = piqa_rows
    _BENCH_CACHE["sciq"] = retry_network(_load_dataset, "allenai/sciq", split="validation")
    _BENCH_CACHE["arc_easy"] = retry_network(_load_dataset, "allenai/ai2_arc", "ARC-Easy", split="validation")
    _BENCH_CACHE["social_iqa"] = retry_network(_load_dataset, "lighteval/siqa", split="validation")
    _BENCH_CACHE["copa"] = retry_network(_load_dataset, "aps/super_glue", "copa", split="validation")
    for _k, _v in _BENCH_CACHE.items():
        BENCH_SIZES[_k] = len(_v)
    print("[Section 5] All 6 benchmark datasets cached:", BENCH_SIZES)
    return _BENCH_CACHE

def score_completion(model_key, active_sent, context, completion):
    tok = get_tok(model_key); model = get_model(model_key); device = MODEL_CFG[model_key]["device"]
    sent = context + " " + completion
    active_sent[0] = sent
    enc = tok(sent, return_tensors="pt", truncation=True, max_length=512).to(device)
    ctx_l = tok(context, return_tensors="pt", truncation=True, max_length=512)["input_ids"].shape[1]
    with torch.no_grad():
        logits = model(**enc).logits
    lp = F.log_softmax(logits[0].float(), dim=-1)
    ids = enc["input_ids"][0]
    scores = [lp[i - 1, ids[i]].item() for i in range(ctx_l, len(ids))]
    return float(np.mean(scores)) if scores else -np.inf

def make_benchmarks(model_key, active_sent):
    sc = lambda c, comp: score_completion(model_key, active_sent, c, comp)
    bench = load_benchmark_datasets()
    def hellaswag(n):
        ds = bench["hellaswag"]
        outcomes = [int(np.argmax([sc(ex["ctx"], e) for e in ex["endings"]]) == int(ex["label"]))
                    for ex in ds.select(range(min(n, len(ds))))]
        return float(np.mean(outcomes)), outcomes
    def piqa(n):
        rows = bench["piqa"][:n]
        outcomes = [int(np.argmax([sc(ex["goal"], ex[f"sol{i+1}"]) for i in range(2)]) == int(ex["label"])) for ex in rows]
        return float(np.mean(outcomes)), outcomes
    def sciq(n):
        ds = bench["sciq"]
        outcomes = []
        for ex in ds.select(range(min(n, len(ds)))):
            choices = [ex["distractor1"], ex["distractor2"], ex["distractor3"], ex["correct_answer"]]
            outcomes.append(int(np.argmax([sc(ex["question"], c) for c in choices]) == 3))
        return float(np.mean(outcomes)), outcomes
    def arc_easy(n):
        ds = bench["arc_easy"]
        outcomes = []
        for ex in ds.select(range(min(n, len(ds)))):
            labels = ex["choices"]["label"]
            if ex["answerKey"] not in labels: continue
            idx = labels.index(ex["answerKey"])
            outcomes.append(int(np.argmax([sc(ex["question"], c) for c in ex["choices"]["text"]]) == idx))
        return (float(np.mean(outcomes)) if outcomes else 0.0), outcomes
    def social_iqa(n):
        ds = bench["social_iqa"]
        outcomes = []
        for ex in ds.select(range(min(n, len(ds)))):
            lbl = int(ex["label"]) - 1
            prompt = ex["context"].strip() + " " + ex["question"].strip()
            choices = [ex["answerA"], ex["answerB"], ex["answerC"]]
            outcomes.append(int(np.argmax([sc(prompt, c) for c in choices]) == lbl))
        return (float(np.mean(outcomes)) if outcomes else 0.0), outcomes
    def copa(n):
        ds = bench["copa"]
        outcomes = []
        for ex in ds.select(range(min(n, len(ds)))):
            conn = "because" if ex["question"] == "cause" else "so"
            ctx = ex["premise"].rstrip(".") + f" {conn}"
            outcomes.append(int(np.argmax([sc(ctx, ex[f"choice{i+1}"]) for i in range(2)]) == ex["label"]))
        return (float(np.mean(outcomes)) if outcomes else 0.0), outcomes
    return [("HellaSwag", hellaswag), ("PIQA", piqa), ("SciQ", sciq),
            ("ARC-Easy", arc_easy), ("Social IQA", social_iqa), ("COPA", copa)]

In [ ]:
import time

def run_downstream(model_key, condition, level, seed):
    """Score all 6 benchmarks under one (condition, level, seed).

    Returns (row, outcomes) where outcomes is {benchmark: [0/1 per example]}.
    Per-example outcomes are kept, not just the mean: every condition is scored on
    the SAME examples, so a paired test is available. Unpaired at N_DOWNSTREAM=200
    a single accuracy has SE ~3.5pp and a difference ~4.9pp -- too coarse for the
    ~5pp effects this figure is about. Paired (McNemar) on the pooled ~1,100
    examples resolves them. The first version of this function computed `outcomes`
    and threw it away.
    """
    cfg = MODEL_CFG[model_key]; fixed = get_fixed(model_key)
    model = get_model(model_key); tok = get_tok(model_key)
    df = best_fits[model_key].sort_values("best_iou", ascending=False).reset_index(drop=True)
    iou_order = [(int(r.layer), int(r.head)) for r in df.itertuples()]
    smart_lookup = {(int(r.layer), int(r.head)): r.best_program for r in df.itertuples()}
    cats = categories_by_model[model_key]
    cat_of = {p: c for c, members in cats.items() for p in members}
    cat_names = list(cats.keys())
    rng = random.Random(seed)
    progs = load_programs(model_key)
    target = int(cfg["total_heads"] * level)
    heads = iou_order[:target]

    zero_heads = None
    if condition == "baseline":
        # level 0.0 -- nothing replaced, nothing zeroed: the untouched model.
        # Run once per model; chosen and zero are identical here by definition.
        assignment = {}
    elif condition == "chosen":
        assignment = {h: smart_lookup[h] for h in heads}
    elif condition == "zero":
        # Same IoU-ordered heads as "chosen", but zeroed out post-value-mixing
        # instead of substituted with a program. This is the control that decides
        # whether the programs are doing work or the heads were simply unimportant.
        assignment = {}
        zero_heads = set(heads)
    else:  # wrong_category
        assignment = {}
        for h in heads:
            own_cat = cat_of.get(smart_lookup[h])
            if own_cat is None:
                assignment[h] = smart_lookup[h]; continue
            other_cats = [c for c in cat_names if c != own_cat]
            assignment[h] = rng.choice(cats[rng.choice(other_cats)]) if other_cats else smart_lookup[h]

    mat_cache = {}
    def get_mat(prog_name, sentence):
        key = (prog_name, sentence)
        if key not in mat_cache:
            try:
                _, mat = progs[prog_name](sentence, tok)
                mat_cache[key] = np.asarray(mat, dtype=np.float64)
            except Exception:
                mat_cache[key] = None
        return mat_cache[key]

    active_sent = [""]
    state = {"assignment": assignment, "pattern": get_mat, "sentence": active_sent}
    if zero_heads is not None:
        state["zero_heads"] = zero_heads
    restore = fixed.install(model, state)
    row, outcomes = {}, {}
    t0 = time.time()
    try:
        for name, fn in make_benchmarks(model_key, active_sent):
            acc, out = fn(N_DOWNSTREAM)
            row[name] = acc
            outcomes[name] = list(out)
    finally:
        restore()  # same _MODEL_CACHE corruption risk as run_ppl_sweep -- 5 of 6
                   # benchmarks call load_dataset() while the model is patched
    row.update({"model": model_key, "condition": condition,
                "replacement_pct": int(round(level * 100)), "seed": seed,
                "secs": round(time.time() - t0, 1)})
    return row, outcomes


# ---- driver -----------------------------------------------------------------
# RESUME SAFETY: checkpoint after every condition, so a crash only costs the
# condition in flight. Keyed on (model, condition, pct, seed).
_checkpoint_path = OUT / "downstream_results_v2.csv"
_outcomes_path   = OUT / "downstream_outcomes_v2.csv"
downstream_rows, outcome_rows, _done = [], [], set()
if _checkpoint_path.exists():
    _prior = pd.read_csv(_checkpoint_path)
    downstream_rows = _prior.to_dict("records")
    _done = set(zip(_prior["model"], _prior["condition"], _prior["replacement_pct"], _prior["seed"]))
    print(f"[Section 5] RESUMING: {len(_done)} conditions already done, loaded from {_checkpoint_path.name}")
if _outcomes_path.exists():
    outcome_rows = pd.read_csv(_outcomes_path).to_dict("records")

_BENCH_COLS = ["HellaSwag", "PIQA", "SciQ", "ARC-Easy", "Social IQA", "COPA"]

def _emit(mk, cond, level, seed):
    pct = int(round(level * 100))
    if (mk, cond, pct, seed) in _done:
        print(f"[{mk}][{cond} {pct}%] SKIPPED (already done)")
        return
    row, outs = run_downstream(mk, cond, level, seed)
    downstream_rows.append(row)
    for b, vals in outs.items():
        for i, c in enumerate(vals):
            outcome_rows.append({"model": mk, "condition": cond, "replacement_pct": pct,
                                 "seed": seed, "benchmark": b, "example_idx": i,
                                 "correct": int(c)})
    pd.DataFrame(downstream_rows).to_csv(_checkpoint_path, index=False)
    pd.DataFrame(outcome_rows).to_csv(_outcomes_path, index=False)
    _done.add((mk, cond, pct, seed))
    accs = "  ".join(f"{k}={row[k]:.3f}" for k in _BENCH_COLS if k in row)
    print(f"[{mk}][{cond} {pct}%] {accs}  ({row['secs']:.0f}s)", flush=True)

_levels = [l for l in DOWNSTREAM_LEVELS if l > 0]
_has_base = any(l == 0.0 for l in DOWNSTREAM_LEVELS)
_conds = ["chosen", "zero"] + (["wrong_category"] if RUN_WRONG_CATEGORY else [])
_per_model = int(_has_base) + len(_levels) * (2 + (N_SEEDS if RUN_WRONG_CATEGORY else 0))
print(f"[Section 5] levels {[f'{int(l*100)}%' for l in _levels]} x {_conds}"
      + (" + 1 baseline" if _has_base else "")
      + f" = {_per_model} conditions/model x 6 benchmarks x {len(MODELS)} model(s)")
if not RUN_WRONG_CATEGORY:
    print("[Section 5] wrong-category arm DISABLED (RUN_WRONG_CATEGORY=False). No seed "
          "averaging is lost -- `seed` is read only in the wrong-category branch; "
          "chosen and zero are deterministic.")
if not _has_base:
    print("[Section 5] WARNING: no 0.0 in DOWNSTREAM_LEVELS -- the figure will have no "
          "unreplaced reference point and 'accuracy preserved' cannot be shown.")

for mk in MODELS:
    if _has_base:
        _emit(mk, "baseline", 0.0, 0)
    for level in _levels:
        _emit(mk, "chosen", level, 0)
        _emit(mk, "zero", level, 0)
        if RUN_WRONG_CATEGORY:
            for seed in range(1, N_SEEDS + 1):
                _emit(mk, "wrong_category", level, seed)
    print(f"[Section 5] {mk} complete.")

downstream_df = pd.DataFrame(downstream_rows)
downstream_df.to_csv(_checkpoint_path, index=False)
outcomes_df = pd.DataFrame(outcome_rows)
outcomes_df.to_csv(_outcomes_path, index=False)
print("Saved:", _checkpoint_path)
print("Saved:", _outcomes_path, f"({len(outcomes_df)} per-example rows)")

In [ ]:
bench_names = ["HellaSwag", "PIQA", "SciQ", "ARC-Easy", "Social IQA", "COPA"]

# ---- paired chosen-vs-X comparison, pooled over all 6 benchmarks -------------
# McNemar's exact test on the per-example outcomes. Valid because every condition
# is scored on identical examples: only the discordant pairs carry information,
# which is what makes ~5pp effects resolvable at N_DOWNSTREAM=200 per benchmark.
from scipy.stats import binomtest

def _outcome_vec(mk, cond, pct, seed=0):
    o = outcomes_df[(outcomes_df.model == mk) & (outcomes_df.condition == cond)
                    & (outcomes_df.replacement_pct == pct) & (outcomes_df.seed == seed)]
    if not len(o):
        return None
    return o.sort_values(["benchmark", "example_idx"])["correct"].to_numpy()

paired_rows = []
if len(outcomes_df):
    for mk in MODELS:
        base = _outcome_vec(mk, "baseline", 0)
        for cond in ("chosen", "zero", "wrong_category"):
            sub_c = outcomes_df[(outcomes_df.model == mk) & (outcomes_df.condition == cond)]
            for pct in sorted(sub_c.replacement_pct.unique()):
                for seed in sorted(sub_c[sub_c.replacement_pct == pct].seed.unique()):
                    v = _outcome_vec(mk, cond, pct, seed)
                    if v is None or base is None or len(v) != len(base):
                        continue
                    b01 = int(((base == 0) & (v == 1)).sum())   # baseline wrong, cond right
                    b10 = int(((base == 1) & (v == 0)).sum())   # baseline right, cond wrong
                    n_disc = b01 + b10
                    p = binomtest(b01, n_disc, 0.5).pvalue if n_disc else 1.0
                    paired_rows.append({
                        "model": mk, "condition": cond, "replacement_pct": int(pct), "seed": int(seed),
                        "n": len(v), "acc_baseline": float(base.mean()), "acc_condition": float(v.mean()),
                        "delta_pp": float((v.mean() - base.mean()) * 100),
                        "discordant": n_disc, "cond_better": b01, "cond_worse": b10,
                        "mcnemar_p": float(p),
                    })
if paired_rows:
    paired_df = pd.DataFrame(paired_rows).sort_values(["model", "condition", "replacement_pct", "seed"])
    paired_df.to_csv(OUT / "downstream_paired_vs_baseline_v2.csv", index=False)
    print("Paired (McNemar) vs unreplaced baseline, pooled over 6 benchmarks:")
    print(paired_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("Saved:", OUT / "downstream_paired_vs_baseline_v2.csv")
else:
    print("[Section 5] no per-example outcomes / no baseline -- skipping paired test.")

# ---- wrong-category seed aggregate (only when that arm actually ran) ---------
agg_rows = []
for mk in MODELS:
    wrong = downstream_df[(downstream_df.model == mk) & (downstream_df.condition == "wrong_category")]
    if not len(wrong):
        continue
    agg = {"model": mk, "n_seeds": int(wrong.seed.nunique())}
    for b in bench_names:
        vals = wrong[b].values
        agg[f"{b}_mean"] = float(np.mean(vals)) if len(vals) else float("nan")
        agg[f"{b}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    agg_rows.append(agg)
if agg_rows:
    agg_df = pd.DataFrame(agg_rows)
    agg_df.to_csv(OUT / "downstream_wrong_category_aggregate_v2.csv", index=False)
    print()
    print(agg_df.to_string(index=False))
    print("Saved:", OUT / "downstream_wrong_category_aggregate_v2.csv")
else:
    print("\n[Section 5] wrong-category arm did not run (RUN_WRONG_CATEGORY=False) -- "
          "skipping its seed aggregate.")

print("\n[DONE] all_experiments_v2.ipynb run complete.")

## Figure 7: Downstream Accuracy vs. Heads Replaced

Six benchmarks. Chosen program and zero-ablation are deterministic (single pass);
the wrong-category baseline is drawn fresh per seed, so it is plotted as
mean +/- 1 std across seeds. Panel sample sizes are read from the loaded
datasets, not hardcoded.

In [ ]:
# -- Figure 7 -- Effect of Replacing Attention Heads on Downstream Evaluations --
# One combined figure across all models, styled to match the rest of the paper
# (serif, the per-model colours used in Figures 4.3/4.4, legend below the panels).
# Solid = chosen program, dashed = zero-ablation, both anchored at the shared 0%
# unreplaced point so "preserved" vs "degraded" is readable off the plot.
import matplotlib.lines as mlines

try:
    _df = downstream_df
except NameError:                      # allow running this cell on its own
    _df = pd.read_csv(OUT / "downstream_results_v2.csv")

PLOTS_DIR = RES_DIR / "plots"; PLOTS_DIR.mkdir(exist_ok=True)
plt.rcParams["font.family"] = "serif"

# N is the size of the split actually evaluated from, so the panels are
# consistent; BENCH_SIZES is empty unless Section 5 ran in this session.
BENCH_META = [
    ("HellaSwag",  "hellaswag",  "Common Sense Inference", 0.25),
    ("PIQA",       "piqa",       "Physical Reasoning",     0.50),
    ("SciQ",       "sciq",       "Science Understanding",  0.25),
    ("ARC-Easy",   "arc_easy",   "Elementary Science",     0.25),
    ("Social IQA", "social_iqa", "Social Interaction",     1 / 3),
    ("COPA",       "copa",       "Causal Reasoning",       0.50),
]
_FALLBACK_N = {"hellaswag": 10042, "piqa": 1838, "sciq": 1000,
               "arc_easy": 570, "social_iqa": 1954, "copa": 100}
MODEL_COLOR = {"gpt2":      {"line": "#c0392b", "box": "#e07070"},
               "tinyllama": {"line": "#1f77b4", "box": "#7fb3d5"},
               "llama3b":   {"line": "#2a9d8f", "box": "#b5ead7"},
               "qwen3":     {"line": "#8e44ad", "box": "#c39bd3"}}

def _fmt_n(n):
    if n is None:  return ""
    return f" ($N={n/1000:.1f}k$)".replace(".0k", "k") if n >= 1000 else f" ($N={n}$)"

_models = [m for m in MODELS if len(_df[_df.model == m])]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.4), constrained_layout=True)
for ax, (col, key, title, chance) in zip(axes.ravel(), BENCH_META):
    ax.axhline(chance, color="black", linestyle="--", dashes=(8, 4), linewidth=1.2)
    for mk in _models:
        c = MODEL_COLOR.get(mk, {"line": "#555555", "box": "#bbbbbb"})
        sub = _df[_df.model == mk]
        base = sub[sub.condition == "baseline"]
        b0 = float(base[col].iloc[0]) if len(base) else None
        for cond, style in (("chosen", dict(ls="-", lw=1.8, marker="o", ms=5,
                                            markerfacecolor=c["box"], zorder=3)),
                            ("zero",   dict(ls="--", lw=1.4, alpha=0.85, zorder=2))):
            cs = sub[sub.condition == cond].sort_values("replacement_pct")
            if not len(cs):
                continue
            xs, ys = cs.replacement_pct.values, cs[col].values
            if b0 is not None:                       # anchor at the shared 0% point
                xs = np.concatenate([[0], xs]); ys = np.concatenate([[b0], ys])
            ax.plot(xs, ys, color=c["line"], **style)
    ax.set_title(col, fontweight="bold", fontsize=11, pad=20)
    ax.text(0.5, 1.01, f"{title}{_fmt_n(BENCH_SIZES.get(key, _FALLBACK_N.get(key)))}",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=8.5, color="#333333")
    ax.set_ylim(0, 1.0); ax.set_xlim(0, 100); ax.set_xticks(range(0, 101, 20))
    ax.set_xlabel("% Attention Heads Replaced", fontsize=10)
    ax.set_ylabel("Accuracy Score", fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

handles = [mlines.Line2D([], [], color="black", ls="--", dashes=(8, 4), lw=1.2,
                         label="Random Guessing")]
for mk in _models:
    c = MODEL_COLOR.get(mk, {"line": "#555555", "box": "#bbbbbb"})
    handles.append(mlines.Line2D([], [], color=c["line"], lw=1.8, marker="o", ms=5,
                                 markerfacecolor=c["box"],
                                 label=f"{MODEL_LABEL.get(mk, mk)} — Chosen"))
for mk in _models:
    c = MODEL_COLOR.get(mk, {"line": "#555555", "box": "#bbbbbb"})
    handles.append(mlines.Line2D([], [], color=c["line"], lw=1.4, ls="--", alpha=0.85,
                                 label=f"{MODEL_LABEL.get(mk, mk)} — Zero-Ablation"))
try:
    fig.legend(handles=handles, loc="outside lower center", ncol=4, frameon=False, fontsize=9)
except ValueError:                     # matplotlib < 3.6 has no "outside" locs
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9,
               bbox_to_anchor=(0.5, -0.04))

for ext, dpi in (("pdf", 600), ("png", 170)):
    plt.savefig(PLOTS_DIR / f"figure_7_downstream_zero.{ext}", dpi=dpi, bbox_inches="tight")
plt.show()
print("Saved:", PLOTS_DIR / "figure_7_downstream_zero.pdf")
